# FHRPY — False-signal detection on the **GPU** (TensorFlow / Keras)

The FS weights were trained in Keras, so a `tf.keras.layers.GRU(reset_after=True)`
loads them **directly** (no gate reordering). This notebook builds the model with
TensorFlow and **self-verifies** the output against the NumPy reference.

> Requires `tensorflow` (`pip install tensorflow`). It runs on the GPU when a
> TF-GPU build is available for your CUDA/GPU, else on CPU. (On very new GPUs a
> matching TF-GPU wheel may not exist yet — the PyTorch notebook is the tested
> GPU path here.)

In [ ]:
import sys, pathlib, importlib.util
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "fhrpy" / "__init__.py").exists()), None)
if _root and str(_root) not in sys.path: sys.path.insert(0, str(_root))
if importlib.util.find_spec("tensorflow") is None:
    %pip install -q tensorflow
import tensorflow as tf, fhrpy
print("fhrpy", fhrpy.__version__, "| tensorflow", tf.__version__,
      "| GPUs", tf.config.list_physical_devices("GPU"))

## Build the TF model and verify it matches NumPy

In [ ]:
import numpy as np
import fhrpy.datasets as ds
from fhrpy.io import read_fhr
from fhrpy.falsesignal.detect import load_model
from fhrpy.falsesignal.features import build_dop_features
from fhrpy.falsesignal.tf_backend import FSDopTF

rec = read_fhr(ds.example_path("ctg_example_01"))
ref = load_model("doppler")(build_dop_features(rec.fhr1, rec.mhr))[0]   # NumPy reference
got = FSDopTF().detect(rec.fhr1, rec.mhr)                               # TensorFlow
diff = float(np.max(np.abs(ref - got)))
print("TF vs NumPy  max|diff| =", diff)
assert diff < 5e-3, "TF output diverges from the NumPy reference!"
print("OK — TensorFlow model matches the reference.")

## Throughput (GPU when available)

In [ ]:
import time
names = ["fs_dopmhr_train0006","fs_dopmhr_train0010","fs_dopmhr_train0022","ctg_example_01"]
recs = [read_fhr(ds.example_path(n)) for n in names]
feats = [build_dop_features(r.fhr1, r.mhr).T for r in recs]
M = load_model("doppler"); T = FSDopTF()
T(feats[0])  # warm up / build graph
t=time.time(); [M(I.T) for I in feats]; t_cpu=time.time()-t
t=time.time(); [T(I) for I in feats]; t_tf=time.time()-t
hours=sum(len(r)/4/3600 for r in recs)
print(f"{len(recs)} records, {hours:.1f} h | NumPy CPU {t_cpu:.1f}s | TensorFlow {t_tf:.1f}s")